In [ ]:
"""
Two-Phase Exam Scheduling Model
================================
Phase 1: Assign exams to timeslots  (y variables only — no rooms)
Phase 2: Assign exams to rooms      (per-timeslot bin-packing)

Same inputs, same constraints, same objective weights as the original
monolithic model, but dramatically fewer variables and constraints.

Original: |E|×|T|×|R| binary x variables  (exam × timeslot × room)
Phase 1:  |E|×|T|      binary y variables  (exam × timeslot)
Phase 2:  per timeslot, only exams assigned to that slot × |R|
"""

import gurobipy as gp
from gurobipy import GRB
from collections import defaultdict
import time


# ─────────────────────────────────────────────────────────────────────────────
#  DUMMY-ROOM DEFINITIONS  (identical to original)
# ─────────────────────────────────────────────────────────────────────────────
# Each dummy room is the combination of two physical rooms.
# When a dummy is used at time t, both constituent rooms are blocked.
DUMMY_ROOM_COMPONENTS = {
    ('dummy', '1'): [('HRZ', 'AMP'),  ('KCK', '100')],
    ('dummy', '2'): [('DCH', '1055'), ('HRG', '100')],
    ('dummy', '3'): [('HRZ', '210'),  ('HRZ', '212')],
}


# ─────────────────────────────────────────────────────────────────────────────
#  PRE-PROCESSING  (shared by both phases)
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(E, T, R, S, N, capacities, class_sizes):
    """Derive all helper structures used by both phases."""
    E_set = set(E)

    # CRNs per schedule that actually have a final exam
    exam_crns_by_sched = {
        idx: [crn for crn in crns if crn in E_set]
        for idx, crns in S.items()
    }

    # Pairwise student-overlap counts (only pairs sharing ≥1 student)
    overlap = defaultdict(int)
    for idx, crns in S.items():
        ec = [crn for crn in crns if crn in E_set]
        for i in range(len(ec)):
            for j in range(i + 1, len(ec)):
                pair = (min(ec[i], ec[j]), max(ec[i], ec[j]))
                overlap[pair] += N[idx]
    E_pairs = list(overlap.keys())

    # Filter schedules by exam count to avoid trivially-satisfied constraints
    sched_keys_2plus = [idx for idx in S if len(exam_crns_by_sched[idx]) >= 2]
    sched_keys_3plus = [idx for idx in S if len(exam_crns_by_sched[idx]) >= 3]

    # Total room capacity per timeslot (for aggregate feasibility in Phase 1)
    total_capacity = sum(capacities[r] for r in R)

    # Timeslot weight: earlier slots are cheaper (index 1…|T|)
    w = {t: i + 1 for i, t in enumerate(T)}

    return dict(
        E_set=E_set,
        exam_crns_by_sched=exam_crns_by_sched,
        overlap=overlap,
        E_pairs=E_pairs,
        sched_keys_2plus=sched_keys_2plus,
        sched_keys_3plus=sched_keys_3plus,
        total_capacity=total_capacity,
        w=w,
    )


# ─────────────────────────────────────────────────────────────────────────────
#  PHASE 1:  TIMESLOT ASSIGNMENT
# ─────────────────────────────────────────────────────────────────────────────
def phase1_timeslot_assignment(E, T, R, S, N, capacities, class_sizes, pre):
    """
    Assign every exam to exactly one timeslot.
    Decision variable:  y[crn, date, time]  ∈ {0,1}

    Hard constraints
    ────────────────
    • Each exam assigned to exactly one timeslot.
    • No student has two exams in the same slot  (conflict-free).
    • Aggregate capacity: total enrolled students in a slot ≤ total room capacity.

    Soft-constraint penalties  (same weights as original)
    ─────────────────────────
    • Timeslot weight:       Σ w[t] · y[e,t]                     (earlier = better)
    • Overlap penalty (λ=1): Σ overlap[c1,c2] · z[c1,c2,t]      (same-slot pairs)
    • 3-in-48h penalty (μ=10): via u indicator variables
    • Back-to-back penalty (ν=5): via d indicator variables
    """
    t0 = time.time()
    env = gp.Env(empty=True)
    env.setParam("OutputFlag", 1)
    env.start()
    m = gp.Model("phase1_timeslots", env=env)

    E_set              = pre['E_set']
    exam_crns_by_sched = pre['exam_crns_by_sched']
    overlap            = pre['overlap']
    E_pairs            = pre['E_pairs']
    sched_keys_2plus   = pre['sched_keys_2plus']
    sched_keys_3plus   = pre['sched_keys_3plus']
    total_capacity     = pre['total_capacity']
    w                  = pre['w']

    # ── Decision variables ────────────────────────────────────────────────
    y = m.addVars(E, T, vtype=GRB.BINARY, name="y")
    z = m.addVars(E_pairs, T, vtype=GRB.CONTINUOUS, name="z")
    print(f"[{time.time()-t0:.1f}s] Phase 1 vars: "
          f"{len(E)*len(T):,} y, {len(E_pairs)*len(T):,} z")

    # ── Hard: each exam exactly once ──────────────────────────────────────
    m.addConstrs(
        (gp.quicksum(y[crn, *t] for t in T) == 1 for crn in E),
        name="assign_once"
    )

    # ── Hard: no student conflict (two exams in same slot) ────────────────
    # For every overlapping pair, they cannot both be in the same slot.
    # This is already implied by minimising z with positive overlap, BUT
    # if overlap penalty λ is small relative to other terms the solver might
    # still place them together.  Making it a hard constraint is safer and
    # mirrors the original model's intent (overlap cost, not a ban — so we
    # keep this as a soft penalty only, matching the original).
    # >>> If you want hard no-conflict, uncomment below:
    # m.addConstrs(
    #     (y[c1, *t] + y[c2, *t] <= 1
    #      for (c1, c2) in E_pairs for t in T),
    #     name="no_conflict"
    # )

    # ── Hard: aggregate capacity per timeslot ─────────────────────────────
    # Sum of enrolled students assigned to slot t ≤ total available seats.
    m.addConstrs(
        (gp.quicksum(class_sizes.get(crn, 0) * y[crn, *t] for crn in E)
         <= total_capacity
         for t in T),
        name="agg_capacity"
    )
    print(f"[{time.time()-t0:.1f}s] Hard constraints added")

    # ── Linearize z[c1,c2,t] = y[c1,t] · y[c2,t] ────────────────────────
    m.addConstrs(
        (z[c1, c2, *t] <= y[c1, *t] for c1, c2 in E_pairs for t in T),
        name="z_ub1"
    )
    m.addConstrs(
        (z[c1, c2, *t] <= y[c2, *t] for c1, c2 in E_pairs for t in T),
        name="z_ub2"
    )
    m.addConstrs(
        (z[c1, c2, *t] >= y[c1, *t] + y[c2, *t] - 1
         for c1, c2 in E_pairs for t in T),
        name="z_lb"
    )
    print(f"[{time.time()-t0:.1f}s] z linearisation: {3*len(E_pairs)*len(T):,} constraints")

    # ── 3-in-48h tracking (v / u) — only for schedules with 3+ exams ─────
    # 3 slots/day → 6-slot window = 48 hours
    window_starts = T[:-5]
    v = m.addVars(window_starts, sched_keys_3plus, vtype=GRB.INTEGER, name="v")
    u = m.addVars(window_starts, sched_keys_3plus, vtype=GRB.BINARY, name="u")

    for t_idx, t_start in enumerate(window_starts):
        window = T[t_idx:t_idx + 6]
        for sidx in sched_keys_3plus:
            crns = exam_crns_by_sched[sidx]
            m.addConstr(
                v[*t_start, sidx] == gp.quicksum(
                    y[crn, *t] for crn in crns for t in window
                ),
                name=f"v_def[{t_start},{sidx}]"
            )
            m.addConstr(
                v[*t_start, sidx] <= 2 + len(crns) * u[*t_start, sidx],
                name=f"u_bigM[{t_start},{sidx}]"
            )
    print(f"[{time.time()-t0:.1f}s] 48h constraints: "
          f"{2*len(window_starts)*len(sched_keys_3plus):,} (v_def + u_bigM)")

    # ── Back-to-back tracking (d) — only for schedules with 2+ exams ─────
    consec_starts = T[:-1]
    d = m.addVars(consec_starts, sched_keys_2plus, vtype=GRB.BINARY, name="d")

    for t_idx, t in enumerate(consec_starts):
        t_next = T[t_idx + 1]
        for sidx in sched_keys_2plus:
            crns = exam_crns_by_sched[sidx]
            at_t      = gp.quicksum(y[crn, *t]      for crn in crns)
            at_t_next = gp.quicksum(y[crn, *t_next] for crn in crns)
            m.addConstr(d[*t, sidx] <= at_t,                 name=f"d_ub1[{t},{sidx}]")
            m.addConstr(d[*t, sidx] <= at_t_next,            name=f"d_ub2[{t},{sidx}]")
            m.addConstr(d[*t, sidx] >= at_t + at_t_next - 1, name=f"d_lb[{t},{sidx}]")
    print(f"[{time.time()-t0:.1f}s] Back-to-back constraints: "
          f"{3*len(consec_starts)*len(sched_keys_2plus):,}")

    # ── Objective (same weights as original) ──────────────────────────────
    lamb = 1
    mu   = 10
    nu   = 5

    m.setObjective(
        # Timeslot weight — sum over y, not x×R  (|R|× fewer terms)
        gp.quicksum(w[t] * y[crn, *t] for crn in E for t in T)
        # Overlap penalty
        + lamb * gp.quicksum(
            overlap[(c1, c2)] * z[c1, c2, *t]
            for c1, c2 in E_pairs for t in T
        )
        # 3-in-48h penalty
        + mu * gp.quicksum(
            u[*t, sidx] for t in window_starts for sidx in sched_keys_3plus
        )
        # Back-to-back penalty
        + nu * gp.quicksum(
            d[*t, sidx] for t in consec_starts for sidx in sched_keys_2plus
        ),
        GRB.MINIMIZE
    )
    print(f"[{time.time()-t0:.1f}s] Objective set")

    # ── Solve ─────────────────────────────────────────────────────────────
    print(f"\n[{time.time()-t0:.1f}s] Phase 1 model built. "
          f"Vars: {m.NumVars:,}  Constrs: {m.NumConstrs:,}  "
          f"Binaries: {m.NumBinVars:,}")

    t_solve = time.time()
    m.optimize()
    print(f"[{time.time()-t0:.1f}s] Phase 1 solved "
          f"(solve: {time.time()-t_solve:.1f}s, "
          f"status: {m.Status})")

    if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
        print("Phase 1 failed — no feasible timeslot assignment found.")
        return None

    print(f"Phase 1 objective: {m.ObjVal:.2f}")

    # Extract timeslot assignments
    timeslot_assignment = {}  # crn -> timeslot tuple
    for crn in E:
        for t in T:
            if y[crn, *t].X > 0.5:
                timeslot_assignment[crn] = t
                break

    return timeslot_assignment


# ─────────────────────────────────────────────────────────────────────────────
#  PHASE 2:  ROOM ASSIGNMENT  (one small IP per timeslot)
# ─────────────────────────────────────────────────────────────────────────────
def phase2_room_assignment(timeslot_assignment, E, T, R, capacities, class_sizes):
    """
    Given fixed timeslot assignments, assign each exam to a room.
    Solved independently per timeslot — each is a small bin-packing IP.

    Hard constraints:
    • Each exam gets exactly one room.
    • Room capacity respected (sum of students in room ≤ capacity).
    • Dummy-room blocking: if a dummy is used, its constituent physical
      rooms cannot be used in the same slot.

    Objective: minimise total wasted capacity (good room utilisation).
    """
    t0 = time.time()
    env = gp.Env(empty=True)
    env.setParam("OutputFlag", 0)
    env.start()

    # Group exams by their assigned timeslot
    exams_in_slot = defaultdict(list)
    for crn, t in timeslot_assignment.items():
        exams_in_slot[t].append(crn)

    # Build physical↔dummy blocking map
    physical_to_dummies = defaultdict(list)  # physical room -> list of dummy rooms containing it
    for dummy, components in DUMMY_ROOM_COMPONENTS.items():
        for phys in components:
            physical_to_dummies[phys].append(dummy)

    room_assignment = {}  # crn -> room tuple
    total_slots = len([t for t in T if exams_in_slot[t]])

    for slot_idx, t in enumerate(T):
        exams = exams_in_slot.get(t, [])
        if not exams:
            continue

        m = gp.Model(f"phase2_slot_{slot_idx}", env=env)

        # x[crn, bldg, room] ∈ {0,1}
        x = m.addVars(exams, R, vtype=GRB.BINARY, name="x")

        # Each exam → exactly one room
        m.addConstrs(
            (gp.quicksum(x[crn, *r] for r in R) == 1 for crn in exams),
            name="one_room"
        )

        # Room capacity: total students assigned to room r ≤ capacity[r]
        m.addConstrs(
            (gp.quicksum(class_sizes.get(crn, 0) * x[crn, *r] for crn in exams)
             <= capacities[r]
             for r in R),
            name="capacity"
        )

        # Dummy-room blocking: if dummy is used, its physical rooms are blocked
        for dummy, components in DUMMY_ROOM_COMPONENTS.items():
            for phys in components:
                if phys in R:
                    m.addConstr(
                        gp.quicksum(x[crn, *phys] for crn in exams)
                        + gp.quicksum(x[crn, *dummy] for crn in exams)
                        <= 1,
                        name=f"block_{dummy}_{phys}"
                    )

        # Objective: minimise wasted capacity (prefer tighter room fits)
        m.setObjective(
            gp.quicksum(
                capacities[r] * x[crn, *r]
                for crn in exams for r in R
            ),
            GRB.MINIMIZE
        )

        m.optimize()

        if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            print(f"  WARNING: slot {t} infeasible — {len(exams)} exams, "
                  f"total students = {sum(class_sizes.get(e,0) for e in exams)}")
            # Fallback: leave unassigned so caller can handle
            for crn in exams:
                room_assignment[crn] = ("UNASSIGNED", "UNASSIGNED")
            continue

        for crn in exams:
            for r in R:
                if x[crn, *r].X > 0.5:
                    room_assignment[crn] = r
                    break

    print(f"[{time.time()-t0:.1f}s] Phase 2 complete — "
          f"{len(room_assignment)} exams assigned to rooms across "
          f"{total_slots} active timeslots")

    return room_assignment


# ─────────────────────────────────────────────────────────────────────────────
#  FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def schedule_exams(E, T, R, S, N, capacities, class_sizes):
    """
    Two-phase exam scheduler.  Returns dict: crn -> {timeslot, room}.
    """
    t0 = time.time()

    print("=" * 70)
    print("  PHASE 1: Timeslot Assignment")
    print("=" * 70)
    pre = preprocess(E, T, R, S, N, capacities, class_sizes)
    print(f"Overlap pairs: {len(pre['E_pairs']):,}  |  "
          f"Schedules w/ 2+ exams: {len(pre['sched_keys_2plus']):,}  |  "
          f"Schedules w/ 3+ exams: {len(pre['sched_keys_3plus']):,}")

    timeslot_assignment = phase1_timeslot_assignment(
        E, T, R, S, N, capacities, class_sizes, pre
    )
    if timeslot_assignment is None:
        return None

    print(f"\n{'=' * 70}")
    print("  PHASE 2: Room Assignment")
    print("=" * 70)
    room_assignment = phase2_room_assignment(
        timeslot_assignment, E, T, R, capacities, class_sizes
    )

    # Combine results
    result = {}
    for crn in E:
        result[crn] = {
            "timeslot": timeslot_assignment[crn],
            "room": room_assignment.get(crn, ("UNASSIGNED", "UNASSIGNED")),
        }

    print(f"\n{'=' * 70}")
    print(f"  DONE — total wall time: {time.time()-t0:.1f}s")
    print(f"  Scheduled {len(result)} exams across {len(T)} timeslots "
          f"and {len(R)} rooms")
    print("=" * 70)

    return result


# ─────────────────────────────────────────────────────────────────────────────
#  ENTRY POINT  (drop-in replacement for original schedule_model call)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # This block mirrors the original notebook's calling convention.
    # Replace the data-loading section with your own (test or real).
    import json as _json

    # ── Paste your data-loading code here, then call: ──
    # result = schedule_exams(E, T, R, S, N, capacities, class_sizes)
    #
    # if result is not None:
    #     output = {
    #         str(crn): {
    #             'date':     v['timeslot'][0],
    #             'time':     v['timeslot'][1],
    #             'building': v['room'][0],
    #             'room':     v['room'][1],
    #         }
    #         for crn, v in result.items()
    #     }
    #     with open('exam_schedule_optimized.json', 'w') as f:
    #         _json.dump(output, f, indent=2)
    #     print('Schedule written.')

    print("Import this module or paste your data-loading code above.")